# M8 · Backfill `fact_fueling`

**Goal:** load fueling events (fuel-ups) joining `staging.document` (header — date, cost, vehicle) and `staging.fueling` (detail — quantity, odometer, payment).

**SQL file:** `sql/24_fact_fueling_incr.sql`. DELETE-INSERT-on-window (no reliable natural key).

**Watermark column:** `document.date_operation`. We filter `doc_type='Fueling'` at load time.

In [1]:
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src)); break

from accent_fleet.config import settings, load_pipeline_config
from accent_fleet.db import get_engine
from sqlalchemy import text
from datetime import datetime, timedelta

s = settings(); cfg = load_pipeline_config()
MIN_TIME = datetime.fromisoformat(cfg['window']['min_event_time'].replace('Z','+00:00')).replace(tzinfo=None)
CHUNK_DAYS = cfg['window']['backfill_chunk_days']
print(f'DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}')

DB = medali_dev@localhost:15432/accent_data


## 2. Inputs

In [2]:
with get_engine().connect() as conn:
    end_time = conn.execute(text("SELECT MAX(date_operation) FROM staging.document WHERE doc_type='Fueling'")).scalar_one()
    n_fuel = conn.execute(text("SELECT COUNT(*) FROM staging.document WHERE doc_type='Fueling'")).scalar_one()
print(f'document rows where doc_type=Fueling: {n_fuel:,}')
print('max date_operation =', end_time)

document rows where doc_type=Fueling: 4,096
max date_operation = 2024-02-10 09:30:00


## 3. Execute

In [3]:
import importlib, time, pandas as pd
import accent_fleet.transforms.facts as facts_module
from accent_fleet.pipeline.run_log import begin_run, end_run
facts_module = importlib.reload(facts_module)
load_fact_incremental = facts_module.load_fact_incremental

run_id = begin_run(mode='notebook:10_load_fact_fueling')
progress, cursor, t0 = [], MIN_TIME, time.time()
try:
    while cursor < end_time:
        nxt = min(cursor + timedelta(days=CHUNK_DAYS), end_time)
        res = load_fact_incremental('fact_fueling', run_id=run_id, window_end=nxt)
        progress.append({'window_start': cursor, 'window_end': nxt, 'rows_loaded': res.rows_loaded})
        cursor = nxt
    end_run(run_id, status='success', rows_loaded=sum(p['rows_loaded'] for p in progress))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc)); raise
print(f'done in {time.time()-t0:.1f}s')
pd.DataFrame(progress).tail(10)

2026-04-29 23:48:42 [info     ] fact_load.start                end=datetime.datetime(2019, 10, 31, 0, 0) fact=fact_fueling run_id=33 start=datetime.datetime(2019, 10, 1, 0, 0)
2026-04-29 23:48:43 [info     ] fact_load.done                 fact=fact_fueling new_watermark=datetime.datetime(2019, 10, 31, 0, 0) rows=344
2026-04-29 23:48:43 [info     ] fact_load.start                end=datetime.datetime(2019, 11, 30, 0, 0) fact=fact_fueling run_id=33 start=datetime.datetime(2019, 10, 30, 23, 50)
2026-04-29 23:48:44 [info     ] fact_load.done                 fact=fact_fueling new_watermark=datetime.datetime(2019, 11, 30, 0, 0) rows=239
2026-04-29 23:48:45 [info     ] fact_load.start                end=datetime.datetime(2019, 12, 30, 0, 0) fact=fact_fueling run_id=33 start=datetime.datetime(2019, 11, 29, 23, 50)
2026-04-29 23:48:45 [info     ] fact_load.done                 fact=fact_fueling new_watermark=datetime.datetime(2019, 12, 30, 0, 0) rows=17
2026-04-29 23:48:46 [info     ] fact_load

,window_start,window_end,rows_loaded
44,2023-05-13,2023-06-12 00:00:00,0
45,2023-06-12,2023-07-12 00:00:00,0
46,2023-07-12,2023-08-11 00:00:00,152
47,2023-08-11,2023-09-10 00:00:00,196
48,2023-09-10,2023-10-10 00:00:00,169
49,2023-10-10,2023-11-09 00:00:00,199
50,2023-11-09,2023-12-09 00:00:00,193
51,2023-12-09,2024-01-08 00:00:00,40
52,2024-01-08,2024-02-07 00:00:00,99
53,2024-02-07,2024-02-10 09:30:00,0


## 4. Inspect

In [4]:
import pandas as pd
with get_engine().connect() as conn:
    total = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_fueling')).scalar_one()
    summary = pd.read_sql(text('''SELECT
        COUNT(*) AS rows_loaded,
        AVG(quantity_l) AS avg_litres,
        AVG(cost_per_litre) AS avg_eur_per_l,
        SUM(cost_total) AS total_cost
      FROM warehouse.fact_fueling'''), conn)
print(f'fact_fueling rows: {total:,}'); display(summary)

fact_fueling rows: 1,648


,rows_loaded,avg_litres,avg_eur_per_l,total_cost
0,1648,1008.032299,1.938901,166825.19
